# 04 - SARIMA / SARIMAX

AIC grid search (p=0..6, d=0..2, q=0..6, then seasonal terms), residual diagnostics, and 24 h forecasts with confidence intervals (Part 4).

The full 147-model grid takes ~20 min; run `python scripts/run_sarima_grid.py --d 0/1/2` then `--seasonal` to reproduce. Results are read from `outputs/metrics/`.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from appliance_energy.config import *
from appliance_energy import data, eda, features, evaluation, plotting
from appliance_energy.models import sarimax as sar

In [ ]:
grid = pd.read_csv(METRICS_DIR / 'arima_grid_full.csv')
print('models fitted:', len(grid))
grid.head(10)

In [ ]:
pd.read_csv(METRICS_DIR / 'sarima_seasonal_grid.csv')

Adding the seasonal component (1,1,1,24) improves AIC by ~545 - overwhelming evidence for daily seasonality. Final model: **SARIMA(1,1,3)(1,1,1,24)**, chosen by AIC with a parsimony rule (fewest parameters within 2 AIC of the minimum).

In [ ]:
hourly = data.resample_hourly(data.load_raw())
y = hourly[TARGET]
train, test = data.train_test_split(y, TEST_STEPS)
fit = sar.fit_sarimax(train, (1, 1, 3), (1, 1, 1, 24))
print(fit.summary())

## Residual diagnostics

Ljung-Box p > 0.05 at lags 24 and 48: residuals are consistent with white noise, i.e. the model has captured the autocorrelation structure. The histogram/Q-Q plots show heavy tails - the Gaussian intervals will be approximate.

In [ ]:
print(sar.residual_diagnostics(fit, 'sarima'))
display(Image(str(FIGURE_DIR / 'fig_resid_sarima.png')))

In [ ]:
mean24, ci24 = sar.forecast_with_ci(fit, HORIZON)
fc = sar.rolling_sarimax_forecast(fit, y, test.index, HORIZON)
evaluation.metrics_table([
    evaluation.evaluate_forecast('sarima', test, fc, train)])